In [1]:
pip install -U transformers datasets evaluate accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 16.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompati

In [2]:
import os
from datasets import load_dataset
from transformers import (
    T5ForConditionalGeneration, T5TokenizerFast,
    DataCollatorForSeq2Seq, TrainingArguments, Trainer
)
import evaluate
import numpy as np

In [10]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer


In [6]:
dataset = load_dataset("knkarthick/samsum")

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [8]:
pip install rouge_score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f5f8358a61cf3f1babde0209d61b039b8761dcd4314e62c1c391e3e2c5e188af
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
from transformers import AutoTokenizer

# Load the T5-small tokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-small")
prefix = "summarize: "

max_input_length = 512
max_target_length = 128

def preprocess_function_samsum(examples):
    # Prepare the inputs (the 'dialogue')
    inputs = [prefix + doc for doc in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")

    # Prepare the "labels" (the 'summary')
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["summary"], max_length=max_target_length, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the preprocessing
tokenized_dataset = dataset.map(preprocess_function_samsum, batched=True)

print("\nTokenizing complete.")
print(tokenized_dataset)

from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Load the T5-small model
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# Define the training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-samsum-results",    # Directory to save the model
    eval_strategy="epoch",         
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    report_to="none",                    
    num_train_epochs=8,                  
    predict_with_generate=True,         
    fp16=True,                           
)

# The data collator dynamically pads sequences to the max length in a batch
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Create the Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Starting T5 training on SAMSum...")
trainer.train()
print("Training finished.")

# Save the final model & tokenizer
trainer.save_model("./t5-samsum-model")

from transformers import pipeline

# Load the summarization pipeline with fine-tuned model
summarizer_pipe = pipeline("summarization", model="./t5-samsum-model", tokenizer=tokenizer)

test_dialogue = dataset["test"][10]["dialogue"]

print("\n--- Testing with a new dialogue ---")
print("\nORIGINAL DIALOGUE:")
print(test_dialogue)

# Generate summary
summary = summarizer_pipe(test_dialogue, max_length=100, min_length=10, do_sample=False)

print("\nGENERATED SUMMARY:")
print(summary[0]['summary_text'])

print("\nACTUAL HUMAN SUMMARY:")
print(dataset["test"][10]["summary"])


from transformers import pipeline, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")
summarizer = pipeline("summarization", model="t5-small", tokenizer=tokenizer)

test_dialogue = """
Wanda: Let's make a party!
Gina: Why?
Wanda: beacuse. I want some fun!
Gina: ok, what do u need?
Wanda: 1st I need too make a list
Gina: noted and then?
Wanda: well, could u take yours father car and go do groceries with me?
Gina: don't know if he'll agree
Wanda: I know, but u can ask :)
Gina: I'll try but theres no promisess
Wanda: I know, u r the best!
Gina: When u wanna go
Wanda: Friday?
Gina: ok, I'll ask
"""

print(summarizer(test_dialogue))

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]


Tokenizing complete.
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})


/tmp/ipython-input-1714449769.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


...

Training finished.


Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Testing with a new dialogue ---

ORIGINAL DIALOGUE:
Wanda: Let's make a party!
Gina: Why?
Wanda: beacuse. I want some fun!
Gina: ok, what do u need?
Wanda: 1st I need too make a list
Gina: noted and then?
Wanda: well, could u take yours father car and go do groceries with me?
Gina: don't know if he'll agree
Wanda: I know, but u can ask :)
Gina: I'll try but theres no promisess
Wanda: I know, u r the best!
Gina: When u wanna go
Wanda: Friday?
Gina: ok, I'll ask

GENERATED SUMMARY:
Gina and Wanda will go to a party on Friday.

ACTUAL HUMAN SUMMARY:
Wanda wants to throw a party. She asks Gina to borrow her father's car and go do groceries together. They set the date for Friday. 


Device set to use cuda:0
Your max_length is set to 200, but your input_length is only 162. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=81)


[{'summary_text': "Gina: ok, what do u need? Wanda: I know, u r the best! u'll ask if u wanna go Wanda ."}]
